### Importar bibliotecas e carregar os dados

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

df = pd.read_csv("../dataset/modified_data.csv")


### Diagnóstico inicial

Antes de alterar os dados, verificamos tipos, campos vazios e valores duplicados

In [ ]:
df = df.replace(r'^\s*$', np.nan, regex=True)
df.isna().sum()


In [ ]:
df.duplicated().sum()

## 3. Padronizar tipos e textos

A coluna `date` é convertida para data, as colunas numéricas são convertidas explicitamente e os campos textuais têm espaços nas extremidades removidos. Conversões inválidas se tornam ausentes e serão tratadas na próxima etapa.

In [ ]:
colunas_numericas = [
    'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
    'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
    'yr_built', 'yr_renovated', 'price_per_sqft'
]
colunas_texto = ['street', 'city', 'statezip']

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df[colunas_numericas] = df[colunas_numericas].apply(pd.to_numeric, errors='coerce')
df[colunas_texto] = df[colunas_texto].apply(lambda column: column.str.strip())
df.dtypes

### Remover registros inválidos

São essenciais para a análise: data, localização, preço, quartos, banheiros, área habitável, área do lote, número de andares, área acima do solo e ano de construção. Removemos registros ausentes nesses campos e valores não positivos onde zero não faz sentido.

In [ ]:
colunas_necessarias = [
    'date', 'street', 'city', 'statezip', 'price', 'bedrooms', 'bathrooms',
    'sqft_living', 'sqft_lot', 'floors', 'sqft_above', 'yr_built'
]

df = df.dropna(subset=colunas_necessarias).drop_duplicates().copy()
valido = (
    (df['price'] > 0)
    & (df['bedrooms'] > 0)
    & (df['bathrooms'] > 0)
    & (df['sqft_living'] > 0)
    & (df['sqft_lot'] > 0)
    & (df['floors'] > 0)
    & (df['sqft_above'] > 0)
    & (df['yr_built'].between(1800, pd.Timestamp.today().year))
)
df = df.loc[valido].copy()
print(f'Registros removidos: {rows_before - len(df)}')

### Recalcular a variável derivada e validar o resultado

`price_per_sqft` é recalculada a partir de `price` e `sqft_living`, evitando que valores derivados antigos ou inconsistentes permaneçam na base. Por fim, confirmamos que não restaram ausências nos campos essenciais.

In [ ]:
df['price_per_sqft'] = (df['price'] / df['sqft_living']).round(2)

assert df[required_columns].isna().sum().sum() == 0, 'Há ausências em campos essenciais.'
assert (df[['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'sqft_above']] > 0).all().all(), 'Há valores inválidos.'

df.head()

### Exportar a base limpa

In [28]:
df.to_csv("../dataset/limpo.csv", index=False)